# ChordSense Pro — Training on Colab
GPU: L4 24GB | Runtime: GPU

## ⚙️ GPU Config — chỉnh tại đây trước khi train

In [ ]:
import os

# --- Tuỳ chỉnh theo GPU đang dùng ---
# T4  16GB  → BATCH_SIZE=16
# L4  24GB  → BATCH_SIZE=32
# A100 40GB → BATCH_SIZE=64
os.environ['BATCH_SIZE']  = '32'    # ← đổi ở đây nếu đổi GPU
os.environ['NUM_WORKERS'] = '4'
os.environ['EPOCHS']      = '20'
os.environ['LR_HEAD']     = '2e-4'
os.environ['SEED']        = '42'
os.environ['FP16']        = 'true'

print('Config overrides set.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
DRIVE_PROJECT = "/content/drive/MyDrive/ChordSensePro"
REPO_DIR = "/content/ChordSensePro"

if not os.path.exists(REPO_DIR):
    !git clone --branch qkhenh https://github.com/qkhenh/ChordSensePro.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin qkhenh

os.chdir(os.path.join(REPO_DIR, "AI"))
!pwd

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
DATA_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v2")
DATA_DST = "/content/data_v2"

if not os.path.exists(DATA_DST):
    !cp -r {DATA_SRC} {DATA_DST}
    print("Data copied.")
else:
    print("Data already exists.")

!ls {DATA_DST}

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path("configs/default.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

cfg["data_dir"] = DATA_DST
cfg["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", "checkpoints")
cfg["results_csv"] = os.path.join(DRIVE_PROJECT, "AI", "results", "results.csv")

colab_cfg_path = "configs/colab.yaml"
with open(colab_cfg_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Config saved to", colab_cfg_path)

In [ ]:
!python -m src.train --config configs/colab.yaml

## Resume from checkpoint

In [ ]:
CKPT = os.path.join(cfg["checkpoint_dir"], "last.pt")
if os.path.exists(CKPT):
    !python -m src.train --config configs/colab.yaml --resume {CKPT}
else:
    print("No checkpoint found.")

## Extract Embeddings (optional — for baseline experiments)

In [ ]:
!python -m src.extract_embeddings --config configs/colab.yaml --out_dir results/embeddings

## Quick Inference Test

In [ ]:
import numpy as np

test_audio_dir = os.path.join(DATA_DST, "test", "audio")
test_files = sorted(os.listdir(test_audio_dir))[:3]

BEST_CKPT = os.path.join(cfg["checkpoint_dir"], "best.pt")

for f in test_files:
    audio_path = os.path.join(test_audio_dir, f)
    !python -m src.infer --config configs/colab.yaml --checkpoint {BEST_CKPT} --audio {audio_path}

## Export ONNX + Benchmark

In [ ]:
!python -m src.export_onnx --config configs/colab.yaml --checkpoint {BEST_CKPT} --output results/model.onnx --benchmark